In [1]:
import numpy as np
from scipy.linalg import block_diag
import itertools
import scipy as sc
import scipy.sparse as scs
from joblib import Parallel, delayed
import sys
import copy

In [2]:
#inverted number of rishons of each link configuration
#0=Down, 1=Up, 2=Right

s0 = np.array([0,0,0])
s1 = np.array([1,1,0])
s2 = np.array([0,1,1])
s3 = np.array([1,0,1])

r = np.array([s0,s1,s2,s3]) #rishon register

In [3]:
def boolean_combinations(n):
    return [
        *itertools.product(
            *[[0, 1, 2, 3] for _ in range(n)]
    )]

In [4]:
def ten2four(n):
    n = n % (4294967296)
    s = np.base_repr(n,4)
    return np.array(list(s.zfill(16)),dtype=int)

In [7]:
def get_plaquete_h():
    W1 = 0
    W2 = 0
    W3_1 = 0
    W3_2 = 0
    W4 = 0
    W5 = 0
    W6 = 0
    W7 = 0
    W8 = 0

    T1 = -128
    T2 = 32
    T3 = 32
    T4 = 32
    T5 = -32
    T6 = -32
    T7 = -32
    T8 = 32

    H1 = np.matrix([[W1, T1],[T1, W1]])
    H2 = np.matrix([[W2, T2],[T2, W2]])
    H3 = np.matrix([[W3_1, T3],[T3.conjugate(), W3_2]])
    H4 = np.matrix([[W4, T4],[T4, W4]])
    H5 = np.matrix([[W5, T5],[T5, W5]])
    H6 = np.matrix([[W6, T6],[T6, W6]])
    H7 = np.matrix([[W7, T7],[T7, W7]])
    H8 = np.matrix([[W8, T8],[T8, W8]])

    H1_s = np.kron(np.eye(1,dtype=int),H1)
    H2_s = np.kron(np.eye(6,dtype=int),H2)
    H3_s = np.kron(np.eye(6,dtype=int),H3)
    H4_s = np.kron(np.eye(3,dtype=int),H4)
    H5_s = np.kron(np.eye(6,dtype=int),H5)
    H6_s = np.kron(np.eye(6,dtype=int),H6)
    H7_s = np.kron(np.eye(3,dtype=int),H7)
    H8_s = np.kron(np.eye(1,dtype=int),H8)

    H_p = block_diag(H1_s, H2_s, H3_s, H4_s, H5_s, H6_s, H7_s, H8_s)
    print(H_p)
    basis_trafo = np.zeros((4,4,4,64))

    ones = [
        (3,1,2,0),
        (0,0,0,1),

        (2,0,0,2),
        (1,1,2,3),
        (0,2,0,4),
        (3,3,2,5),
        (0,3,0,6),
        (3,2,2,7),
        (0,0,3,8),
        (3,1,1,9),
        (0,0,1,10),
        (3,1,3,11),
        (1,0,0,12),
        (2,1,2,13),

        (2,2,0,14),
        (1,3,2,15),
        (0,1,0,16),
        (3,0,2,17),
        (0,3,3,18),
        (3,2,1,19),
        (0,0,2,20),
        (3,1,0,21),
        (1,0,1,22),
        (2,1,3,23),
        (3,0,0,24),
        (0,1,2,25),

        (2,1,0,26),
        (1,0,2,27),
        (0,1,3,28),
        (3,0,1,29),
        (0,3,2,30),
        (3,2,0,31),

        (2,3,0,32),
        (1,2,2,33),
        (0,2,3,34),
        (3,3,1,35),
        (0,3,1,36),
        (3,2,3,37),
        (1,0,3,38),
        (2,1,1,39),
        (2,0,1,40),
        (1,1,3,41),
        (1,2,0,42),
        (2,3,2,43),

        (2,3,3,44),
        (1,2,1,45),
        (0,2,2,46),
        (3,3,0,47),
        (1,3,1,48),
        (2,2,3,49),
        (3,0,3,50),
        (0,1,1,51),
        (2,2,1,52),
        (1,3,3,53),
        (1,1,0,54),
        (2,0,2,55),

        (2,0,3,56),
        (1,1,1,57),
        (0,2,1,58),
        (3,3,3,59),
        (1,3,0,60),
        (2,2,2,61),

        (2,3,1,62),
        (1,2,3,63),
    ]

    for i in ones:
        basis_trafo[i]=1


    prod1 = np.tensordot(basis_trafo,H_p,axes=(3,0))
    H_trafo_tensor = np.tensordot(prod1, basis_trafo, axes=(3,3)) # can be flattened as above

    return H_trafo_tensor

In [8]:
def get_plaquete_sparse():
    row = []
    column = []
    val = []

    h = get_plaquete_h()

    keys = boolean_combinations(3)

    for i in keys:
        for j in keys:
            a = h[i][j]
            if a != 0:
                val.append(a)
                row.append(i)
                column.append(j)
    return np.array(val), np.array(row), np.array(column)

In [10]:
gauge_indices = np.loadtxt("gauge_indices_red.txt").astype(dtype='uint32')
gauge_indices_base4 = np.array([ten2four(i) for i in gauge_indices])

In [11]:
def index_finder(index_4,i,j,k):
    cond = (gauge_indices_base4[:,i]==index_4[0])
    cond = (gauge_indices_base4[:,j]==index_4[1])*cond
    cond = (gauge_indices_base4[:,k]==index_4[2])*cond

    return np.where(cond==True)

In [12]:
def generate_space_h_mag(p):
    p_indices = np.array([
        (0, 1, 4), (4,  5,  8), ( 8,  9, 12),
        (1, 6, 5), (5, 10,  9), ( 9, 14, 13),
        (2, 3, 6), (6,  7, 10), (10, 11, 14),
        (3, 4, 7), (7,  8, 11), (11, 12, 15),
    ])
    vals = []
    rows = []
    cols = []

    p_val, p_row, p_column = get_plaquete_sparse()
    i,j,k = p_indices[p]
    for v in range(64):
        temp_row = index_finder(p_row[v],i,j,k)[0]
        temp_col = index_finder(p_column[v],i,j,k)[0]
        l = len(temp_row)
        if l>0:
            vals.append(np.ones(l)*p_val[v])
            rows.append(temp_row)
            cols.append(temp_col)
    
    return scs.coo_array((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(8192, 8192))

In [13]:
results = Parallel(n_jobs=22)(delayed(generate_space_h_mag)(i) for i in range(12))

In [14]:
h_mag = np.sum(results)

In [15]:
scs.save_npz("h_mag",h_mag)